In [1]:
# load the required libraries
import re
import os
import numpy as np
import pandas as pd
import h5py 
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.cm as cm
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
import math
from scipy.optimize import curve_fit
from scipy.stats import t
from scipy.ndimage import gaussian_filter1d
from scipy import stats

## Config

In [2]:
# define stimulus list
## define stimulus list
#### 2025-04-21_EGCG
stimulus_lists_0421_EGCG = {
    'w1': ['c1_1', 'c1_2', 'c1_3', 'c1_4', 'c1_5'] * 3,
    'w2': ['c1_1', 'c1_2', 'c1_3', 'c1_4', 'c1_5'] * 3,
    'w3': ['c1_1', 'c1_2', 'c1_3', 'c1_4', 'c1_5'] * 3,
    'w4': ['c1_1', 'c1_2', 'c1_3', 'c1_4', 'c1_5'] * 3,
    'w5': ['c1_1', 'c1_2', 'c1_3', 'c1_4', 'c1_5'] * 3,
    'w6': ['c1_5', 'c1_4', 'c1_3', 'c1_2', 'c1_1'] * 3,  # Reversed and reorder the delta_F_over_F0
}
#### 2025-04-21_EGCG_high
stimulus_lists_0421_EGCG_high = {
    'w1': ['c1_4', 'c1_5', 'c1_6'] * 3,
    'w2': ['c1_4', 'c1_5', 'c1_6'] * 3,
}
#### 2025-0428_TF
stimulus_lists_0428_TF = {
    'w1': ['c3_1', 'c3_2', 'c3_3', 'c3_4', 'c3_5'] * 3,
    'w2': ['c3_1', 'c3_2', 'c3_3', 'c3_4', 'c3_5'] * 3,
    'w3': ['c3_1', 'c3_2', 'c3_3', 'c3_4', 'c3_5'] * 3,
    'w4': ['c3_1', 'c3_2', 'c3_3', 'c3_4', 'c3_5'] * 3,
    'w5': ['c3_1', 'c3_2', 'c3_3', 'c3_4', 'c3_5'] * 3,
    'w6': ['c3_1', 'c3_2', 'c3_3', 'c3_4', 'c3_5'] * 3,
    'w7': ['c3_1', 'c3_2', 'c3_3', 'c3_4', 'c3_5'] * 3,
}

#### 2025-0428_L_Theanine
stimulus_lists_0428_L_Theanine = {
    'w1': ['c1_3', 'c1_6', 'c1_3', 'c1_6', 'c3_3', 'c3_3', 'c2_2', 'c2_2', 'c4_1', 'c4_1'],
    'w2': ['c1_3', 'c1_6', 'c1_3', 'c1_6', 'c3_3', 'c3_3', 'c2_2', 'c2_2', 'c4_1', 'c4_1'],
    'w3': ['c1_3', 'c1_6', 'c1_3', 'c1_6', 'c3_3', 'c3_3', 'c2_2', 'c2_2', 'c4_1', 'c4_1'],
    'w4': ['c1_3', 'c1_6', 'c1_3', 'c1_6', 'c4_1', 'c4_1', 'c3_3', 'c3_3', 'c2_2', 'c2_2'],
    'w5': ['c1_3', 'c1_6', 'c1_3', 'c1_6', 'c4_1', 'c4_1', 'c3_3', 'c3_3', 'c2_2', 'c2_2'],
    'w6': ['c1_3', 'c1_6', 'c1_3', 'c1_6', 'c4_1', 'c4_1', 'c3_3', 'c3_3', 'c2_2', 'c2_2'],
}

In [3]:
# set folder_path
# folder_0421_EGCG = r"G:\LAB\DATA\result\20250421_EGCG"
# folder_0421_EGCG_high = r"G:\LAB\DATA\result\20250421_EGCG_high"
# folder_0428_TF = r"G:\LAB\DATA\result\20250428_TF"
# folder_0428_L_Theanine = r"G:\LAB\DATA\result\20250428_The"
folder_0421_EGCG = r"H:\Process_temporary\WJH\olfactory\ID\result\20250421_EGCG"
folder_0421_EGCG_high = r"H:\Process_temporary\WJH\olfactory\ID\result\20250421_EGCG_high"
folder_0428_TF = r"H:\Process_temporary\WJH\olfactory\ID\result\20250428_TF"
folder_0428_L_Theanine = r"H:\Process_temporary\WJH\olfactory\ID\result\20250428_The"

## Load worm data from infer result and ID annotation
- get stimulus interval info from output_volumes.xlsx
- get ID data from ID.xlsx
- get intensity data from HDF5 file
- detrend and save data in a dictionary

In [4]:
from data_load.get_stimulus_info import extract_intervals_from_excel
from data_load.load_worm_data import load_worm_ID, process_worm_data
# 0421_EGCG
info_excel = os.path.join(folder_0421_EGCG, 'output_volumes.xlsx')
experiment_info_0421_EGCG = extract_intervals_from_excel(info_excel)
ID_EGCG = load_worm_ID(folder_0421_EGCG + r'\ID0421EGCG.xlsx')

In [5]:
worm_data_0421_EGCG = {}
with h5py.File(folder_0421_EGCG + r'\20250421_EGCG.h5', 'r') as f:
    for key in f.keys():
        if key == 'w2' or key == 'w3':
            worm_data_0421_EGCG[key] = process_worm_data(
                key, f, experiment_info_0421_EGCG, ID_EGCG, 
                stimulus_lists=stimulus_lists_0421_EGCG
            )

In [6]:
# 0428_Theanine
info_excel = os.path.join(folder_0428_L_Theanine, 'output_volumes.xlsx')
experiment_info_0428_L_Theanine = extract_intervals_from_excel(info_excel)
ID_Theanine = load_worm_ID(folder_0428_L_Theanine + r'\ID0428_The.xlsx')

In [7]:
## For 2025_0428_L_Theanine
worm_data_0428_L_Theanine = {}
with h5py.File(folder_0428_L_Theanine + r'\20250428_The.h5', 'r') as f:
    for key in f.keys():
        if key == 'w2' or key == 'w3':
            worm_data_0428_L_Theanine[key] = process_worm_data(
                key, f, experiment_info_0428_L_Theanine,
                ID_Theanine, stimulus_lists_0428_L_Theanine,
            )
        elif key == 'w4' or key == 'w5' or key == 'w6':
            # 处理w4的特殊情况
            worm_data_0428_L_Theanine[key] = process_worm_data(
                key, f, experiment_info_0428_L_Theanine,
                ID_Theanine, stimulus_lists_0428_L_Theanine,
                stimulus_sort=[0, 1, 2, 3, 6, 7, 8,9,4,5], buffer_sort=[0, 1, 2, 3,4,7,8,9,10,5,6],
                need_sorting=True
            )

## Divide wormdata into segments
- 1 segment represent 40s data(30vols before stimulus and 120vols after stimulus, stimulus intervals are set to 50 vols)
- segment data is saved as dictionary

In [8]:
from data_load.process_worm_data import extract_neuron_groups

neuron_segment_dict_0428_L_Theanine, neuron_groups_0428_L_Theanine = extract_neuron_groups(worm_data_0428_L_Theanine)

In [9]:
from data_load.process_worm_data import per_worm_zscore
neuron_segments_dict_0428_L_Theanine,_ = per_worm_zscore(neuron_segment_dict_0428_L_Theanine)

In [10]:
from data_load.process_worm_data import reorganize_neuron_segments
neuron_segments_dict_0428_L_Theanine_reorganized = reorganize_neuron_segments(
    neuron_segment_dict_0428_L_Theanine, 
    date='2025-04-28',
)


In [11]:
from data_load.process_worm_data import merge_multiple_dicts
all_neuron_segments = merge_multiple_dicts(neuron_segments_dict_0428_L_Theanine_reorganized)

In [12]:
import copy
neuron_segments_dict = copy.deepcopy(all_neuron_segments)
neuron_groups = copy.deepcopy(neuron_groups_0428_L_Theanine)

In [18]:
neuron_groups

{'ADLL': {'ADLL'},
 'ASGL': {'ASGL'},
 'ASHL': {'ASHL'},
 'ASIL': {'ASIL'},
 'AWBL': {'AWBL'},
 'AWCL': {'AWCL'},
 'ASKL': {'ASKL'},
 'ASJL': {'ASJL'},
 'ASJR': {'ASJR'},
 'ASHR': {'ASHR'},
 'ASIR': {'ASIR'},
 'ASKR': {'ASKR'},
 'ADFR': {'ADFR'},
 'AWAL': {'AWAL'},
 'ASEL': {'ASEL'},
 'ADFL': {'ADFL'},
 'AWAR': {'AWAR'},
 'ADLR': {'ADLR'},
 'AWBR': {'AWBR'},
 'ASER': {'ASER'}}

In [21]:
keys_to_copy = ['ASHL', 'AWBL', 'ASKL', 'ASJL', 'ASJR', 'ASHR', 'ASKR', 'ADFR', 'AWAL', 'ASEL', 'ADFL', 'AWAR', 'AWBR', 'ASER']
neuron_segments_dict = {key: copy.deepcopy(neuron_segments_dict[key]) for key in keys_to_copy}
neuron_groups = {key: copy.deepcopy(neuron_groups[key]) for key in keys_to_copy}

## plot

In [13]:
import json
with open(r'H:\Process_temporary\WJH\sensory_pipeline_python\data_load\config\compound_info.json', 'r') as f:
    compounds = json.load(f)

In [38]:
from result_plot.lineplot import plot_neurons_to_pdf
output_pdf = r'H:\Process_temporary\WJH\olfactory\ID\result\20250428_The\neuron_segments_1.pdf'
plot_neurons_to_pdf(neuron_segments_dict, compounds,output_pdf, if_combine=True)


PDF saved to H:\Process_temporary\WJH\olfactory\ID\result\20250428_The\neuron_segments_1.pdf


'H:\\Process_temporary\\WJH\\olfactory\\ID\\result\\20250428_The\\neuron_segments_1.pdf'

In [22]:
from result_plot.radar_heatmap_plot import compare_compounds_and_dilutions, heatmap_plot
output_folder = r"H:\Process_temporary\WJH\olfactory\ID\result\20250428_The\visulization_7_select"
os.makedirs(output_folder, exist_ok=True)
compare_compounds_and_dilutions(neuron_segments_dict, compounds, output_folder, if_combine=True, if_sum_normalization=True)

After combining L/R neurons: 7 neurons
No data found for compound caffeine
No data found for compound caffeine


(<Figure size 1200x1000 with 1 Axes>,
 {'c1': <Figure size 1200x1000 with 1 Axes>,
  'c2': <Figure size 1200x1000 with 1 Axes>,
  'c3': <Figure size 1200x1000 with 1 Axes>,
  'c4': <Figure size 1200x1000 with 1 Axes>,
  'c5': None},
 None)

In [23]:
heatmap_plot(neuron_segments_dict, compounds, output_folder, if_combine=False)

In [17]:
def calculate_time_to_segment_half_peak(
    neuron_segments_dict, 
    stim_onset_in_segment=30, 
    stim_duration_for_peak_detection=50
):
    """
    Calculates the time it takes for each neuron's response in each segment (trial)
    to reach half of that segment's peak value, measured from stimulus onset.

    Args:
        neuron_segments_dict (dict): Dictionary with structure 
                                     {neuron_id: {stimulus_type: [segment_data, ...]}}.
                                     Each segment_data is a dict with 'deltaFoverF_0' (np.array).
        stim_onset_in_segment (int): The index within each segment's 'deltaFoverF_0'
                                     trace where the stimulus begins.
        stim_duration_for_peak_detection (int): The duration (in frames) from stimulus
                                                onset used to determine the segment's peak response.

    Returns:
        dict: A dictionary with the same structure as neuron_segments_dict, but
              containing the time (in frames from stim_onset_in_segment) to reach
              half-peak for each segment. Values will be None if half-peak is not
              reached or if the peak is zero.
    """
    time_to_half_peak_results = {}

    for neuron_id, stimulus_data in neuron_segments_dict.items():
        time_to_half_peak_results[neuron_id] = {}
        for stimulus_type, segments in stimulus_data.items():
            time_to_half_peak_results[neuron_id][stimulus_type] = []
            
            for segment in segments:
                trace = segment.get('deltaFoverF_0')
                time_to_reach = None

                if trace is None or len(trace) == 0:
                    time_to_half_peak_results[neuron_id][stimulus_type].append(None)
                    continue

                # Ensure trace is long enough for peak detection period
                stim_peak_end_idx = stim_onset_in_segment + stim_duration_for_peak_detection
                if len(trace) < stim_peak_end_idx:
                    time_to_half_peak_results[neuron_id][stimulus_type].append(None)
                    # print(f"Warning: Trace too short for neuron {neuron_id}, stim {stimulus_type}")
                    continue

                # Determine the peak response within the specified stimulus window for this segment
                stim_period_trace_for_peak = trace[stim_onset_in_segment : stim_peak_end_idx]
                
                if len(stim_period_trace_for_peak) == 0:
                    time_to_half_peak_results[neuron_id][stimulus_type].append(None)
                    continue

                max_pos_in_stim = np.max(stim_period_trace_for_peak)
                min_neg_in_stim = np.min(stim_period_trace_for_peak)

                segment_peak_response = 0
                if abs(max_pos_in_stim) >= abs(min_neg_in_stim):
                    segment_peak_response = max_pos_in_stim
                else:
                    segment_peak_response = min_neg_in_stim
                
                if segment_peak_response == 0:
                    time_to_half_peak_results[neuron_id][stimulus_type].append(None)
                    continue

                half_peak_target = segment_peak_response / 2.0
                
                # Search for half-peak from stimulus onset onwards in the segment trace
                trace_to_search = trace[stim_onset_in_segment:]
                indices_reaching_target = []

                if segment_peak_response > 0:
                    # For positive peaks, find first time >= half_peak_target
                    potential_indices = np.where(trace_to_search >= half_peak_target)[0]
                    if len(potential_indices) > 0:
                        indices_reaching_target = potential_indices
                else: # segment_peak_response < 0
                    # For negative peaks, find first time <= half_peak_target
                    potential_indices = np.where(trace_to_search <= half_peak_target)[0]
                    if len(potential_indices) > 0:
                        indices_reaching_target = potential_indices
                
                if len(indices_reaching_target) > 0:
                    time_to_reach = indices_reaching_target[0] 
                    # This index is relative to the start of trace_to_search, 
                    # which is stim_onset_in_segment. So, it's the time from stimulus onset.
                
                time_to_half_peak_results[neuron_id][stimulus_type].append(time_to_reach)
                
    return time_to_half_peak_results

In [18]:
time_to_half_peak_results = calculate_time_to_segment_half_peak(neuron_segments_dict)

In [19]:
time_to_half_peak_results

{'ADFL': {'c1_3': [0, 0, 20, 5],
  'c1_6': [0, 0, 20, 14],
  'c3_3': [4, 0, 31, 2],
  'c2_2': [0, 0, 23, 2],
  'c4_1': [35, 24, 14, 7]},
 'ADFR': {'c1_3': [10, 14],
  'c1_6': [13, 0],
  'c3_3': [14, 14],
  'c2_2': [4, 3],
  'c4_1': [4, 3]},
 'ADLL': {'c1_3': [20, 2, 20, 25, 5, 0, 7, 1],
  'c1_6': [10, 10, 27, 12, 7, 1, 2, 47],
  'c3_3': [3, 0, 0, 0, 21, 3, 6, 0],
  'c2_2': [20, 0, 3, 1, 20, 2, 7, 29],
  'c4_1': [0, 11, 6, 27, 19, 4, 16, 16]},
 'ADLR': {'c1_3': [15, 23, 24, 0],
  'c1_6': [0, 14, 20, 25],
  'c3_3': [2, 0, 16, 0],
  'c2_2': [0, 5, 0, 0],
  'c4_1': [0, 0, 31, 0]},
 'ASEL': {'c1_3': [3, 3],
  'c1_6': [4, 4],
  'c3_3': [14, 16],
  'c2_2': [4, 4],
  'c4_1': [4, 3]},
 'ASER': {'c1_3': [16, 20, 3, 3],
  'c1_6': [19, 15, 27, 0],
  'c3_3': [4, 3, 11, 12],
  'c2_2': [11, 10, 11, 14],
  'c4_1': [4, 4, 18, 3]},
 'ASGL': {'c1_3': [3, 20, 0, 5, 3, 0, 1, 2],
  'c1_6': [3, 0, 3, 3, 0, 7, 6, 5],
  'c3_3': [0, 1, 0, 26, 7, 3, 16, 6],
  'c2_2': [0, 7, 0, 0, 2, 0, 11, 6],
  'c4_1': [18, 9, 

In [26]:
import dash
from dash import dcc, html, Input, Output, State, callback
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import numpy as np
from scipy import stats
import pandas as pd

def create_neuronal_dashboard(neuron_segments_dict, odor_information=None):
    """
    Create a comprehensive Dash app for visualizing neuronal responses.
    
    Parameters:
    -----------
    neuron_segments_dict : dict
        Dictionary with structure {neuron_group: {stimulus_type: [trial data]}}
    odor_information : dict, optional
        Dictionary mapping stimulus codes to descriptions
    """
    # Extract all available data options
    all_neurons = sorted(neuron_segments_dict.keys())
    
    # Extract all unique stimuli
    all_stimuli = set()
    for neuron in neuron_segments_dict:
        all_stimuli.update(neuron_segments_dict[neuron].keys())
    all_stimuli = sorted(all_stimuli)
    
    # Extract all unique worms
    all_worms = set()
    for neuron in neuron_segments_dict:
        for stim in neuron_segments_dict[neuron]:
            for segment in neuron_segments_dict[neuron][stim]:
                if 'worm_key' in segment:
                    all_worms.add(segment['worm_key'])
    all_worms = sorted(all_worms)
    
    # Extract all unique dates
    all_dates = set()
    for neuron in neuron_segments_dict:
        for stim in neuron_segments_dict[neuron]:
            for segment in neuron_segments_dict[neuron][stim]:
                if 'date' in segment:
                    all_dates.add(segment['date'])
    all_dates = sorted(all_dates)
    
    # Create color mappings
    stimulus_color_map = create_color_map(all_stimuli)
    worm_color_map = create_color_map(all_worms, 'Set2')

    # Preprocess data for faster plotting
    preprocessed_data = preprocess_data_for_plotting(neuron_segments_dict)
    
    # Initialize Dash app
    app = dash.Dash(__name__, suppress_callback_exceptions=True)
    app.title = "Neuronal Response Visualization"
    
    # Create app layout with tabs for different visualization types
    app.layout = html.Div([
        html.H1("Neuronal Response Visualization", style={'textAlign': 'center', 'marginBottom': '20px'}),
        
        dcc.Tabs([
            # Tab 1: Individual responses
            dcc.Tab(label="Individual Responses", children=[
                html.Div([
                    html.Div([
                        html.Label("Select Neurons:"),
                        dcc.Dropdown(
                            id='neuron-selector',
                            options=[{'label': neuron, 'value': neuron} for neuron in all_neurons],
                            value=[all_neurons[0]] if all_neurons else [],
                            multi=True
                        ),
                    ], style={'width': '48%', 'display': 'inline-block', 'marginRight': '2%'}),
                    
                    html.Div([
                        html.Label("Select Stimuli:"),
                        dcc.Dropdown(
                            id='stimuli-selector',
                            options=[{'label': get_stimulus_label(s, odor_information), 'value': s} for s in all_stimuli],
                            value=[all_stimuli[0]] if all_stimuli else [],
                            multi=True
                        ),
                    ], style={'width': '48%', 'display': 'inline-block'}),
                    
                    html.Div([
                        html.Label("Filter by Worm:"),
                        dcc.Dropdown(
                            id='worm-selector',
                            options=[{'label': w, 'value': w} for w in all_worms],
                            value=all_worms,
                            multi=True
                        ),
                    ], style={'width': '48%', 'display': 'inline-block', 'marginRight': '2%', 'marginTop': '10px'}),
                    
                    html.Div([
                        html.Label("Filter by Date:"),
                        dcc.Dropdown(
                            id='date-selector',
                            options=[{'label': d, 'value': d} for d in all_dates],
                            value=all_dates,
                            multi=True
                        ),
                    ], style={'width': '48%', 'display': 'inline-block', 'marginTop': '10px'}),
                    
                    html.Div([
                        html.Label("Data Type:"),
                        dcc.RadioItems(
                            id='data-type-selector',
                            options=[
                                {'label': 'Raw ΔF/F₀', 'value': 'deltaFoverF_0'},
                                {'label': 'Z-scored', 'value': 'z_scored'},
                                {'label': 'Scaled', 'value': 'scaled_data'}
                            ],
                            value='deltaFoverF_0',
                            inline=True
                        ),
                    ], style={'marginTop': '10px'}),
                    
                    html.Div([
                        html.Label("Display:"),
                        dcc.RadioItems(
                            id='display-type-selector',
                            options=[
                                {'label': 'Individual Trials', 'value': 'individual'},
                                {'label': 'Mean ± SEM', 'value': 'mean_sem'}
                            ],
                            value='mean_sem',
                            inline=True
                        ),
                    ], style={'marginTop': '10px'}),
                    
                    html.Button('Update Plot', id='update-individual-button', 
                               style={'marginTop': '20px', 'padding': '10px 20px'}),
                    
                ], style={'padding': '20px'}),
                
                dcc.Graph(id='individual-response-plot', style={'height': '800px'})
            ]),
            
            # Tab 2: Grouped responses (averages across neurons)
            dcc.Tab(label="Average Responses", children=[
                html.Div([
                    html.Div([
                        html.Label("Select Neurons:"),
                        dcc.Dropdown(
                            id='neuron-group-selector',
                            options=[{'label': neuron, 'value': neuron} for neuron in all_neurons],
                            value=[all_neurons[0]] if all_neurons else [],
                            multi=True
                        ),
                    ], style={'width': '48%', 'display': 'inline-block', 'marginRight': '2%'}),
                    
                    html.Div([
                        html.Label("Select Stimuli:"),
                        dcc.Dropdown(
                            id='stimuli-group-selector',
                            options=[{'label': get_stimulus_label(s, odor_information), 'value': s} for s in all_stimuli],
                            value=[s for s in all_stimuli if s.startswith('c1_')] if all_stimuli else [],
                            multi=True
                        ),
                    ], style={'width': '48%', 'display': 'inline-block'}),
                    
                    html.Div([
                        html.Label("Data Type:"),
                        dcc.RadioItems(
                            id='data-type-group-selector',
                            options=[
                                {'label': 'Raw ΔF/F₀', 'value': 'deltaFoverF_0'},
                                {'label': 'Z-scored', 'value': 'z_scored'},
                                {'label': 'Scaled', 'value': 'scaled_data'}
                            ],
                            value='deltaFoverF_0',
                            inline=True
                        ),
                    ], style={'marginTop': '10px'}),
                    
                    html.Button('Update Plot', id='update-group-button', 
                               style={'marginTop': '20px', 'padding': '10px 20px'}),
                    
                ], style={'padding': '20px'}),
                
                dcc.Graph(id='group-response-plot', style={'height': '800px'})
            ]),
        ]),
    ])
    
    # Callback for individual responses tab
    @app.callback(
        Output('individual-response-plot', 'figure'),
        Input('update-individual-button', 'n_clicks'),
        [State('neuron-selector', 'value'),
         State('stimuli-selector', 'value'),
         State('worm-selector', 'value'),
         State('date-selector', 'value'),
         State('data-type-selector', 'value'),
         State('display-type-selector', 'value')]
    )
    def update_individual_plot(n_clicks, selected_neurons, selected_stimuli, selected_worms, 
                              selected_dates, data_type, display_type):
        if not selected_neurons or not selected_stimuli:
            return go.Figure()
        
        fig = make_subplots(
            rows=len(selected_neurons),
            cols=len(selected_stimuli),
            shared_yaxes=True,
            horizontal_spacing=0.05,
            vertical_spacing=0.05,
            subplot_titles=[get_stimulus_label(s, odor_information) for s in selected_stimuli]
        )
        
        # Set appropriate y-axis range based on data type
        y_ranges = {
            'deltaFoverF_0': [-0.2, 0.5],
            'z_scored': [-2, 5],
            'scaled_data': [-0.2, 1.2]
        }
        y_range = y_ranges.get(data_type, [-0.2, 0.5])
        
        for row_idx, neuron in enumerate(selected_neurons, 1):
            # Row label
            fig.add_annotation(
                text=neuron,
                x=-0.05, 
                y=0.5,
                xref='paper',
                yref=f'y{row_idx}',
                showarrow=False,
                font=dict(size=14)
            )
            
            for col_idx, stim in enumerate(selected_stimuli, 1):
                # Add stimulus highlighting
                highlight_color = stimulus_color_map.get(stim, 'gray')
                fig.add_shape(
                    type="rect",
                    x0=0, x1=50,
                    y0=y_range[0], y1=y_range[1],
                    fillcolor=highlight_color,
                    opacity=0.15,
                    layer="below",
                    line_width=0,
                    row=row_idx, col=col_idx
                )
                
                # Skip if this combination doesn't exist
                if neuron not in preprocessed_data or stim not in preprocessed_data[neuron]:
                    continue
                
                # Filter segments by worm and date
                segments = [
                    seg for seg in preprocessed_data[neuron][stim] 
                    if seg['worm_key'] in selected_worms
                    and seg['date'] in selected_dates
                    and data_type in seg
                ]
                
                if not segments:
                    continue
                
                if display_type == 'individual':
                    # Plot individual trials
                    for seg in segments:
                        fig.add_trace(
                            go.Scatter(
                                x=seg['x_values'],
                                y=seg[data_type],
                                mode='lines',
                                line=dict(color=worm_color_map[seg['worm_key']], width=1.5),
                                opacity=0.6,
                                showlegend=False,
                                hovertemplate=f"Worm: {seg['worm_key']}<br>Date: {seg['date']}"
                            ),
                            row=row_idx, col=col_idx
                        )
                else:
                    # Calculate and plot mean ± SEM
                    all_data = np.array([seg[data_type] for seg in segments])
                    mean_data = np.mean(all_data, axis=0)
                    sem_data = stats.sem(all_data, axis=0)
                    x_values = segments[0]['x_values']
                    
                    # Plot mean line
                    fig.add_trace(
                        go.Scatter(
                            x=x_values,
                            y=mean_data,
                            mode='lines',
                            line=dict(color='black', width=2),
                            name=f"{stim} (n={len(segments)})",
                            showlegend=(row_idx == 1 and col_idx == 1)
                        ),
                        row=row_idx, col=col_idx
                    )
                    
                    # Plot SEM band
                    fig.add_trace(
                        go.Scatter(
                            x=np.concatenate([x_values, x_values[::-1]]),
                            y=np.concatenate([mean_data + sem_data, (mean_data - sem_data)[::-1]]),
                            fill='toself',
                            fillcolor='rgba(128, 128, 128, 0.4)',
                            line=dict(color='rgba(255,255,255,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=row_idx, col=col_idx
                    )
                    
                    # Add sample size annotation
                    fig.add_annotation(
                        text=f"n={len(segments)}",
                        x=0.95, y=0.95,
                        xref=f'x{col_idx}', yref=f'y{row_idx}',
                        showarrow=False,
                        font=dict(size=10)
                    )
                
                # Set axes ranges and titles
                fig.update_yaxes(range=y_range, row=row_idx, col=col_idx)
                
                # Only add x-label on bottom row
                if row_idx == len(selected_neurons):
                    fig.update_xaxes(title_text="Time (frames)", row=row_idx, col=col_idx)
        
        # Update layout
        fig.update_layout(
            height=300 * len(selected_neurons),
            title="Neuronal Responses to Stimuli",
            template="plotly_white",
            hovermode="closest",
            margin=dict(l=80, r=20, t=50, b=50),
        )
        
        return fig
    
    # Callback for group responses tab
    @app.callback(
        Output('group-response-plot', 'figure'),
        Input('update-group-button', 'n_clicks'),
        [State('neuron-group-selector', 'value'),
         State('stimuli-group-selector', 'value'),
         State('data-type-group-selector', 'value')]
    )
    def update_group_plot(n_clicks, selected_neurons, selected_stimuli, data_type):
        if not selected_neurons or not selected_stimuli:
            return go.Figure()
        
        fig = make_subplots(
            rows=len(selected_neurons),
            cols=len(selected_stimuli),
            shared_yaxes=True,
            horizontal_spacing=0.01,
            vertical_spacing=0.05,
        )
        
        # Set appropriate y-axis range based on data type
        y_ranges = {
            'deltaFoverF_0': [-2, 5],
            'z_scored': [-2, 5],
            'scaled_data': [-0.2, 1.2]
        }
        y_range = y_ranges.get(data_type, [-0.2, 0.5])
        
        for row_idx, neuron in enumerate(selected_neurons, 1):
            # Add row title (neuron name)
            fig.add_annotation(
                text=neuron,
                x=-0.05, y=0.5,
                xref='paper',
                yref=f'y{row_idx}',
                showarrow=False,
                font=dict(size=14)
            )
            
            for col_idx, stim in enumerate(selected_stimuli, 1):
                # Add column title (stimulus name)
                if row_idx == 1:
                    fig.add_annotation(
                        text=get_stimulus_label(stim, odor_information),
                        x=0.5, y=1.1,
                        xref=f'x{col_idx}',
                        yref='paper',
                        showarrow=False,
                        font=dict(size=12)
                    )
                
                # Add stimulus highlighting
                highlight_color = stimulus_color_map.get(stim, 'gray')
                fig.add_shape(
                    type="rect",
                    x0=0, x1=50,
                    y0=y_range[0], y1=y_range[1],
                    fillcolor=highlight_color,
                    opacity=0.15,
                    layer="below",
                    line_width=0,
                    row=row_idx, col=col_idx
                )
                
                # Skip if this combination doesn't exist
                if neuron not in preprocessed_data or stim not in preprocessed_data[neuron]:
                    continue
                
                # Get all segments with this data type
                segments = [seg for seg in preprocessed_data[neuron][stim] if data_type in seg]
                
                if not segments:
                    continue
                
                # Calculate mean and SEM
                all_data = np.array([seg[data_type] for seg in segments])
                mean_data = np.mean(all_data, axis=0)
                sem_data = stats.sem(all_data, axis=0)
                x_values = segments[0]['x_values']
                
                # Plot mean line
                fig.add_trace(
                    go.Scatter(
                        x=x_values,
                        y=mean_data,
                        mode='lines',
                        line=dict(color='black', width=2),
                        showlegend=False
                    ),
                    row=row_idx, col=col_idx
                )
                
                # Plot SEM band
                fig.add_trace(
                    go.Scatter(
                        x=np.concatenate([x_values, x_values[::-1]]),
                        y=np.concatenate([mean_data + sem_data, (mean_data - sem_data)[::-1]]),
                        fill='toself',
                        fillcolor='rgba(128, 128, 128, 0.4)',
                        line=dict(color='rgba(255,255,255,0)'),
                        showlegend=False,
                        hoverinfo='none'
                    ),
                    row=row_idx, col=col_idx
                )
                
                # Add sample size annotation
                fig.add_annotation(
                    text=f"n={len(segments)}",
                    x=0.95, y=0.95,
                    xref=f'x{col_idx}', yref=f'y{row_idx}',
                    showarrow=False,
                    font=dict(size=10)
                )
                
                # Set axes ranges
                fig.update_yaxes(range=y_range, row=row_idx, col=col_idx)
                
                # Only add x-label on bottom row
                if row_idx == len(selected_neurons):
                    fig.update_xaxes(title_text="Time (vols)", row=row_idx, col=col_idx)
        
        # Create color legend for stimuli
        legend_traces = []
        for stim in selected_stimuli:
            legend_traces.append(
                go.Scatter(
                    x=[None], y=[None],
                    mode='markers',
                    marker=dict(color=stimulus_color_map.get(stim, 'gray'), size=10),
                    name=get_stimulus_label(stim, odor_information),
                    legendgroup=stim
                )
            )
        
        # Add legend traces to the first subplot
        for lt in legend_traces:
            fig.add_trace(lt, row=1, col=1)
        
        # Update layout
        fig.update_layout(
            height=300 * len(selected_neurons),
            title="Average Neuronal Responses by Stimulus Type",
            template="plotly_white",
            hovermode="closest",
            legend=dict(x=1.02, y=1.0, bordercolor="Black", borderwidth=1),
            margin=dict(l=80, r=120, t=50, b=50)
        )
        
        return fig
    
    return app

def create_color_map(items, colorscale='Dark2'):
    """Create a color mapping for the given items."""
    import plotly.express as px
    
    # Access colorscale directly as an attribute instead of using get()
    if hasattr(px.colors.qualitative, colorscale):
        colors = getattr(px.colors.qualitative, colorscale)
    else:
        # Default to Plotly colorscale if requested one doesn't exist
        colors = px.colors.qualitative.Plotly
    
    return {item: colors[i % len(colors)] for i, item in enumerate(items)}

def get_stimulus_label(stimulus_code, odor_information=None):
    """Convert stimulus code to readable label using odor_information if available."""
    if odor_information and stimulus_code in odor_information:
        return f"{stimulus_code}: {odor_information[stimulus_code]}"
    return stimulus_code

def preprocess_data_for_plotting(neuron_segments_dict):
    """
    Preprocess neuron segments data for faster plotting.
    
    Returns:
    --------
    dict: Preprocessed data with structure {neuron: {stimulus: [segment_data]}}
          where segment_data includes 'x_values', 'deltaFoverF_0', 'z_scored', etc.
    """
    preprocessed = {}
    
    for neuron, stimuli in neuron_segments_dict.items():
        preprocessed[neuron] = {}
        
        for stim, segments in stimuli.items():
            preprocessed[neuron][stim] = []
            
            for seg in segments:
                processed_seg = {}
                
                # Copy metadata
                for key in ['worm_key', 'segment_index', 'date']:
                    if key in seg:
                        processed_seg[key] = seg[key]
                    else:
                        processed_seg[key] = 'unknown'
                
                # Process time series data
                for data_key in ['deltaFoverF_0', 'z_scored', 'scaled_data']:
                    if data_key in seg and seg[data_key] is not None:
                        data = seg[data_key]
                        processed_seg[data_key] = data
                        
                        # Only compute x_values once
                        if 'x_values' not in processed_seg:
                            processed_seg['x_values'] = np.arange(len(data)) - 30
                
                if 'x_values' in processed_seg:  # Only add if we have some data
                    preprocessed[neuron][stim].append(processed_seg)
    
    return preprocessed

In [ ]:
# Usage example:
app = create_neuronal_dashboard(neuron_segments_dict, compounds)
app.run_server(debug=True, port=8050)

In [36]:
def create_neuronal_dashboard(neuron_segments_dict, odor_information=None):
    """
    Create a streamlined Dash app for visualizing neuronal responses.
    
    Parameters:
    -----------
    neuron_segments_dict : dict
        Dictionary with structure {neuron_group: {stimulus_type: [trial data]}}
    odor_information : dict, optional
        Dictionary mapping stimulus codes to descriptions
    """
    # Extract all available data options
    all_neurons = sorted(neuron_segments_dict.keys())
    
    # Extract all unique stimuli
    all_stimuli = set()
    for neuron in neuron_segments_dict:
        all_stimuli.update(neuron_segments_dict[neuron].keys())
    all_stimuli = sorted(all_stimuli)
    
    # Create color mappings
    stimulus_color_map = create_color_map(all_stimuli)
    
    # Preprocess data for faster plotting
    preprocessed_data = preprocess_data_for_plotting(neuron_segments_dict)
    
    # Initialize Dash app
    app = dash.Dash(__name__, suppress_callback_exceptions=True)
    
    # Create app layout
    app.layout = html.Div([
        html.Div([
            html.Div([
                html.Label("Select Neurons:"),
                dcc.Checklist(
                    id='neuron-selector',
                    options=[{'label': neuron, 'value': neuron} for neuron in all_neurons],
                    value=[all_neurons[0]] if all_neurons else [],
                    labelStyle={'display': 'inline-block', 'margin-right': '10px'}
                ),
            ], style={'marginBottom': '15px'}),
            
            html.Div([
                html.Label("Select Stimuli:"),
                dcc.Checklist(
                    id='stimuli-selector',
                    options=[{'label': get_stimulus_label(s, odor_information), 'value': s} for s in all_stimuli],
                    value=[all_stimuli[0]] if all_stimuli else [],
                    labelStyle={'display': 'inline-block', 'margin-right': '10px'}
                ),
            ], style={'marginBottom': '15px'}),
            
            html.Div([
                html.Label("Display:"),
                dcc.RadioItems(
                    id='display-type-selector',
                    options=[
                        {'label': 'Individual Trials', 'value': 'individual'},
                        {'label': 'Mean ± SEM', 'value': 'mean_sem'}
                    ],
                    value='mean_sem',
                    inline=True
                ),
            ], style={'marginBottom': '15px'}),
            
            html.Div([
                dcc.Checklist(
                    id='combine-options',
                    options=[
                        {'label': 'Combine compounds (different dilutions)', 'value': 'combine_compounds'},
                        {'label': 'Combine neurons (L/R)', 'value': 'combine_neurons'}
                    ],
                    value=[],
                    labelStyle={'display': 'block', 'marginBottom': '5px'}
                ),
            ], style={'marginBottom': '15px'}),
            
            html.Button('Update Plot', id='update-plot-button', 
                       style={'marginTop': '10px', 'padding': '10px 20px'}),
            
        ], style={'padding': '15px', 'backgroundColor': '#f8f9fa', 'borderRadius': '5px'}),
        
        dcc.Graph(id='response-plot', style={'height': '800px'})
    ])
    
    @app.callback(
        Output('response-plot', 'figure'),
        Input('update-plot-button', 'n_clicks'),
        [State('neuron-selector', 'value'),
         State('stimuli-selector', 'value'),
         State('display-type-selector', 'value'),
         State('combine-options', 'value')]
    )
    def update_plot(n_clicks, selected_neurons, selected_stimuli, display_type, combine_options):
        if not selected_neurons or not selected_stimuli:
            return go.Figure()
        
        # Process combination options
        combine_compounds = 'combine_compounds' in combine_options
        combine_neurons = 'combine_neurons' in combine_options
        
        # Apply neuron combination if needed
        processed_neuron_dict = neuron_segments_dict
        if combine_neurons:
            processed_neuron_dict = combine_lr_neurons(neuron_segments_dict)
            # Update selected neurons list based on combined neurons
            # First create a mapping from original neuron names to combined names
            neuron_mapping = create_neuron_mapping(neuron_segments_dict)
            
            # Then update the selected neurons list
            new_selected_neurons = []
            for neuron in selected_neurons:
                if neuron in neuron_mapping:
                    if neuron_mapping[neuron] not in new_selected_neurons:
                        new_selected_neurons.append(neuron_mapping[neuron])
                elif neuron in processed_neuron_dict:
                    new_selected_neurons.append(neuron)
            
            selected_neurons = new_selected_neurons if new_selected_neurons else [list(processed_neuron_dict.keys())[0]]
        
        # Process stimuli combination if needed
        grouped_stimuli = {}
        if combine_compounds:
            # Group stimuli by compound type (e.g., c1_1, c1_2 → c1)
            for stim in selected_stimuli:
                compound = stim.split('_')[0]
                if compound not in grouped_stimuli:
                    grouped_stimuli[compound] = []
                grouped_stimuli[compound].append(stim)
        else:
            # Use stimuli as is
            for stim in selected_stimuli:
                grouped_stimuli[stim] = [stim]
        
        # Calculate dynamic plot dimensions
        base_height_per_neuron = 180
        min_total_height = 600
        total_height = max(min_total_height, len(selected_neurons) * base_height_per_neuron)
        
        base_width_per_stimulus = 200
        min_total_width = 800
        total_width = max(min_total_width, len(grouped_stimuli) * base_width_per_stimulus)
        
        # Create figure with subplots
        fig = make_subplots(
            rows=len(selected_neurons),
            cols=len(grouped_stimuli),
            shared_yaxes=False,  # Allow different y-ranges per neuron
            horizontal_spacing=0.02,
            vertical_spacing=0.08,  # Increased vertical spacing to reduce overlap
        )
        
        # Calculate y-ranges for each neuron
        neuron_y_ranges = {}
        for row_idx, neuron in enumerate(selected_neurons, 1):
            min_values = []
            max_values = []
            
            for group_key, stim_list in grouped_stimuli.items():
                # Collect all data for this neuron across all selected stimuli
                for stim in stim_list:
                    if neuron in processed_neuron_dict and stim in processed_neuron_dict[neuron]:
                        for seg in processed_neuron_dict[neuron][stim]:
                            values = seg['deltaFoverF_0']
                            min_values.append(np.min(values))
                            max_values.append(np.max(values))
            
            if min_values and max_values:
                # Calculate appropriate y-range with buffer
                y_min = min(min_values)
                y_max = max(max_values)
                # Add 15% buffer on each side
                buffer = (y_max - y_min) * 0.15
                neuron_y_ranges[neuron] = [y_min - buffer, y_max + buffer]
            else:
                # Default range if no data
                neuron_y_ranges[neuron] = [-0.2, 0.5]
        
        # Plot each neuron
        for row_idx, neuron in enumerate(selected_neurons, 1):
            # Use neuron-specific y-range
            y_range = neuron_y_ranges[neuron]
            
            # Add neuron label with improved positioning
            fig.add_annotation(
                text=f"<b>{neuron}</b>",
                xref="paper", yref=f"y{row_idx}",
                x=-0.02, y=0.5,  # Moved label position slightly to the left
                showarrow=False,
                font=dict(size=11),  # Reduced font size
                textangle=0,
                align="right"
            )
            
            for col_idx, (group_key, stim_list) in enumerate(grouped_stimuli.items(), 1):
                # Add stimulus highlighting
                highlight_color = stimulus_color_map.get(group_key, 'gray')
                if not combine_compounds:
                    highlight_color = stimulus_color_map.get(stim_list[0], 'gray')
                
                fig.add_shape(
                    type="rect",
                    x0=0, x1=50,
                    y0=y_range[0], y1=y_range[1],  # Use neuron-specific y-range
                    fillcolor=highlight_color,
                    opacity=0.15,
                    layer="below",
                    line_width=0,
                    row=row_idx, col=col_idx
                )
                
                # Stimulus title at top
                stim_label = group_key
                if not combine_compounds:
                    stim_label = get_stimulus_label(stim_list[0], odor_information)
                else:
                    stim_label = f"{group_key} ({len(stim_list)} dilutions)"
                    
                if row_idx == 1:
                    fig.add_annotation(
                        text=f"<b>{stim_label}</b>",
                        xref=f"x{col_idx}", yref="paper",
                        x=0.5, y=1.05,
                        showarrow=False,
                        font=dict(size=12)
                    )
                
                # Collect all segments for this neuron and stimulus group
                all_segments = []
                for stim in stim_list:
                    if neuron in processed_neuron_dict and stim in processed_neuron_dict[neuron]:
                        all_segments.extend(processed_neuron_dict[neuron][stim])
                
                if not all_segments:
                    continue
                    
                if display_type == 'individual':
                    # Plot individual traces
                    for seg in all_segments:
                        values = seg['deltaFoverF_0']
                        x_values = np.arange(len(values)) - 30
                        
                        fig.add_trace(
                            go.Scatter(
                                x=x_values,
                                y=values,
                                mode='lines',
                                line=dict(width=1, color=highlight_color),
                                opacity=0.4,
                                showlegend=False
                            ),
                            row=row_idx, col=col_idx
                        )
                else:
                    # Calculate and plot mean ± SEM
                    all_data = np.array([seg['deltaFoverF_0'] for seg in all_segments])
                    # Make sure all arrays have the same length
                    min_len = min(len(data) for data in all_data)
                    all_data = np.array([data[:min_len] for data in all_data])
                    
                    mean_data = np.mean(all_data, axis=0)
                    sem_data = stats.sem(all_data, axis=0)
                    x_values = np.arange(min_len) - 30
                    
                    # Plot mean line
                    fig.add_trace(
                        go.Scatter(
                            x=x_values,
                            y=mean_data,
                            mode='lines',
                            line=dict(color='black', width=2),
                            showlegend=False
                        ),
                        row=row_idx, col=col_idx
                    )
                    
                    # Plot SEM band
                    fig.add_trace(
                        go.Scatter(
                            x=np.concatenate([x_values, x_values[::-1]]),
                            y=np.concatenate([mean_data + sem_data, (mean_data - sem_data)[::-1]]),
                            fill='toself',
                            fillcolor='rgba(128, 128, 128, 0.4)',
                            line=dict(color='rgba(255,255,255,0)'),
                            showlegend=False,
                            hoverinfo='none'
                        ),
                        row=row_idx, col=col_idx
                    )
                    
                    # Add sample size annotation
                    fig.add_annotation(
                        text=f"n={len(all_data)}",
                        x=0.95, y=0.95,
                        xref=f'x{col_idx}', yref=f'y{row_idx}',
                        showarrow=False,
                        font=dict(size=9, color="gray")
                    )
                
                # Set neuron-specific y-range
                fig.update_yaxes(range=y_range, row=row_idx, col=col_idx, showgrid=False)
                fig.update_xaxes(showgrid=False, row=row_idx, col=col_idx)
                
                # Only add x-label on bottom row
                if row_idx == len(selected_neurons):
                    fig.update_xaxes(title_text="Time (frames)", row=row_idx, col=col_idx)
        
        # Update layout
        fig.update_layout(
            height=total_height,
            width=total_width,
            margin=dict(l=70, r=10, t=30, b=40),  # Increased left margin for neuron labels
            showlegend=False,
            template="plotly_white",
            hovermode="closest"
        )
        
        return fig
    
    return app

def create_neuron_mapping(neuron_segments_dict):
    """
    Create a mapping from original neuron names to combined names.
    For L/R pairs, maps to base name (e.g., ADLL -> ADL, ADLR -> ADL).
    For others, maps to original name.
    """
    mapping = {}
    
    # Find all neurons that might form L/R pairs
    lr_candidates = {}
    for neuron in neuron_segments_dict.keys():
        if neuron.endswith('L') or neuron.endswith('R'):
            base_name = neuron[:-1]  # Remove the L or R suffix
            if base_name not in lr_candidates:
                lr_candidates[base_name] = []
            lr_candidates[base_name].append(neuron)
    
    # Create mappings for neurons that have both L and R versions
    for base_name, neurons in lr_candidates.items():
        if len(neurons) == 2:
            has_left = any(n.endswith('L') for n in neurons)
            has_right = any(n.endswith('R') for n in neurons)
            
            if has_left and has_right:
                for neuron in neurons:
                    mapping[neuron] = base_name
    
    return mapping

def combine_lr_neurons(neuron_segments_dict):
    """
    Combine left and right neuron pairs (e.g., ADLL and ADLR become ADL).
    Only combines neurons that have both L and R versions.
    Returns a new dictionary with combined neurons.
    """
    combined_dict = {}
    
    # Find neurons with L/R suffix
    neuron_groups = {}
    for neuron in neuron_segments_dict:
        if neuron.endswith('L') or neuron.endswith('R'):
            base_name = neuron[:-1]  # Remove the L or R suffix
            if base_name not in neuron_groups:
                neuron_groups[base_name] = []
            neuron_groups[base_name].append(neuron)
    
    # Combine neuron pairs
    for base_name, neurons in neuron_groups.items():
        if len(neurons) == 2:  # If we have both L and R versions
            # Verify one ends with L and one with R
            has_left = any(n.endswith('L') for n in neurons)
            has_right = any(n.endswith('R') for n in neurons)
            
            if has_left and has_right:
                left = next((n for n in neurons if n.endswith('L')), None)
                right = next((n for n in neurons if n.endswith('R')), None)
                
                if left and right:
                    # Create new entry for the combined neuron
                    combined_dict[base_name] = {}
                    
                    # Find common stimuli
                    left_stimuli = set(neuron_segments_dict[left].keys())
                    right_stimuli = set(neuron_segments_dict[right].keys())
                    common_stimuli = left_stimuli.intersection(right_stimuli)
                    
                    # Process each stimulus
                    for stim in common_stimuli:
                        combined_dict[base_name][stim] = []
                        
                        # Get segments from both neurons
                        left_segments = neuron_segments_dict[left][stim]
                        right_segments = neuron_segments_dict[right][stim]
                        
                        # Take the minimum number of segments
                        num_segments = min(len(left_segments), len(right_segments))
                        
                        # Average corresponding segments
                        for i in range(num_segments):
                            left_seg = left_segments[i]
                            right_seg = right_segments[i]
                            
                            # Create a new segment with averaged data
                            combined_seg = {}
                            
                            # Copy metadata from either neuron
                            for key in ['worm_key', 'date', 'segment_index']:
                                combined_seg[key] = left_seg.get(key, right_seg.get(key, 'unknown'))
                            
                            # Average the time series data
                            if 'deltaFoverF_0' in left_seg and 'deltaFoverF_0' in right_seg:
                                left_data = left_seg['deltaFoverF_0']
                                right_data = right_seg['deltaFoverF_0']
                                
                                # Make sure they're the same length
                                min_len = min(len(left_data), len(right_data))
                                left_data = left_data[:min_len]
                                right_data = right_data[:min_len]
                                
                                # Average the data
                                combined_seg['deltaFoverF_0'] = np.mean([left_data, right_data], axis=0)
                                
                                combined_dict[base_name][stim].append(combined_seg)
                    
                    # Add any stimuli that only one neuron has
                    all_stimuli = left_stimuli.union(right_stimuli)
                    for stim in all_stimuli - common_stimuli:
                        source_neuron = left if stim in left_stimuli else right
                        combined_dict[base_name][stim] = neuron_segments_dict[source_neuron][stim]
            else:
                # Keep individual neurons that don't have a pair
                for neuron in neurons:
                    combined_dict[neuron] = neuron_segments_dict[neuron]
        else:
            # Keep individual neurons that don't have a pair
            for neuron in neurons:
                combined_dict[neuron] = neuron_segments_dict[neuron]
    
    # Include neurons that don't have L/R suffix
    for neuron in neuron_segments_dict:
        if not (neuron.endswith('L') or neuron.endswith('R')):
            combined_dict[neuron] = neuron_segments_dict[neuron]
    
    return combined_dict

In [ ]:
app = create_neuronal_dashboard(neuron_segments_dict, compounds)
app.run_server(debug=True, port=8050)